# Spatial Broad-Region Validation — Part 1 (Allen ISH, broad region) + Part 2 (Internal spatial organization)

This notebook consolidates the broad-region (CTX/CNU/IB) re-analysis:

- **Part 1**: Region-level Spearman/Pearson comparison against Allen ISH, using Allen's official
  CTX (Cerebral cortex) / CNU (Cerebral nuclei) / IB (Interbrain) hierarchy instead of the original
  8-region `region_proxy_from_bin_class` labels. Supportive evidence only — region-level statistical
  power is inherently limited (n=3 regions). Sequencing-depth is diagnosed (Supplementary) but no
  correction is applied.
- **Part 2**: Internal spatial validation that does not rely on the Allen atlas — Moran's I / Geary's C
  computed on the whole tissue (bin-level, K=5/10/15), for both the intersect TF set and each method's
  full regulon set.

All output figures/tables are saved under `spatial_broad_region/Part1/` and `spatial_broad_region/Part2/`.


## Part 0. Shared Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import sparse
from scipy.stats import pearsonr, spearmanr, wilcoxon
from sklearn.neighbors import NearestNeighbors

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    matplotlib.use('Agg')

plt.rcParams['figure.dpi'] = 120
np.random.seed(0)

BASE = Path('/Users/kimchaeyoung/Desktop/thesis_data')
H5AD = BASE / 'spagrn/spagrn_spagrn.h5ad'
OUT_ROOT = BASE / 'spatial_broad_region'
OUT_P1 = OUT_ROOT / 'Part1'
OUT_P1_BOXPLOTS = OUT_P1 / 'boxplots'
OUT_P1_DEPTH = OUT_P1 / 'bin_depth_diagnostic'
OUT_P1_TF = OUT_P1 / 'intersectTF_spatial'
OUT_P2 = OUT_ROOT / 'Part2'
OUT_P2_ORIG = OUT_P2 / 'original_region_spatial'
OUT_P2_BROAD = OUT_P2 / 'broad_region_spatial'
OUT_P2_TF = OUT_P2 / 'intersectTF_spatial_moran'
for d in [OUT_P1_BOXPLOTS, OUT_P1_DEPTH, OUT_P1_TF, OUT_P2_ORIG, OUT_P2_BROAD, OUT_P2_TF]:
    d.mkdir(parents=True, exist_ok=True)

METHOD_ORDER = ['GRNBoost2', 'SpaGRN', 'SpaGTL']
METHOD_COLORS = {'GRNBoost2': '#2a78d6', 'SpaGRN': '#008300', 'SpaGTL': '#eb6834', 'Allen': '#888888'}
MIN_TARGETS = 10
K_VALUES = [5, 10, 15]
N_PERM = 999
RANDOM_SEED = 42

print('Setup 완료.')


### 0.1 AnnData 로딩 — 좌표, 원래 region_proxy, broad region(CTX/CNU/IB) 매핑

In [ ]:
adata = ad.read_h5ad(H5AD)
coords = adata.obsm['spatial']
n_spots = adata.n_obs

region_proxy = adata.obs['region_proxy_from_bin_class'].astype(str)
total_counts = adata.obs['total_counts'].values.astype(float)
log_total_counts = np.log1p(total_counts)

# Allen official hierarchy mapping (verified structure_id: CTX=688, CNU=623, IB=1129, graph_id=1)
BROAD_MAP = {
    'Isocortex': 'CTX', 'CTXsp': 'CTX', 'HPF': 'CTX', 'OLF': 'CTX',
    'STR': 'CNU', 'PAL': 'CNU', 'TH': 'IB', 'HY': 'IB',
}
broad_region = region_proxy.map(BROAD_MAP)  # NaN for MB/CB/MY/Pons/NA_region/Unresolved -- excluded downstream
REGIONS = ['CTX', 'CNU', 'IB']
BROAD_STRUCTURE_IDS = {'CTX': 688, 'CNU': 623, 'IB': 1129}

print('adata shape:', adata.shape)
print(region_proxy.value_counts())
print()
print('broad_region counts:')
print(broad_region.value_counts())


### 0.2 Regulon target 목록 + 공식 AUCell 행렬

In [ ]:
def load_gmt_regulons(path):
    targets = {}
    with open(path) as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            tf = parts[0].replace('(+)', '')
            genes = [g for g in parts[2:] if g]
            targets[tf] = set(genes)
    return targets


def load_official_aucell(path, barcode_order):
    df = pd.read_csv(path)
    df = df.set_index('Cell')
    df.columns = [c.replace('(+)', '') for c in df.columns]
    return df.reindex(barcode_order)


REGULON_TARGETS = {
    'GRNBoost2': load_gmt_regulons(BASE / 'd0/aucell_occurrence_ge05/gb2_occ_ge05_regulons.gmt'),
    'SpaGTL': load_gmt_regulons(BASE / 'spagtl/aucell_occurrence_ge05/spagtl_occ_ge05_regulons.gmt'),
}
spagrn_edges = pd.read_csv(BASE / 'spagrn/regulon_to_edge.csv')
spagrn_edges.columns = [c.strip() for c in spagrn_edges.columns]
REGULON_TARGETS['SpaGRN'] = {tf: set(g['target']) for tf, g in spagrn_edges.groupby('TF')}

AUCELL = {
    'GRNBoost2': load_official_aucell(BASE / 'd0/aucell_occurrence_ge05/gb2_occ_ge05_aucell.csv', adata.obs_names),
    'SpaGTL': load_official_aucell(BASE / 'spagtl/aucell_occurrence_ge05/spagtl_occ_ge05_aucell.csv', adata.obs_names),
    'SpaGRN': load_official_aucell(BASE / 'spagrn/auc_mtx_full.csv', adata.obs_names),
}

FULL_TF_LISTS = {}
for m in METHOD_ORDER:
    sizes = {tf: len(t) for tf, t in REGULON_TARGETS[m].items()}
    aucell_tfs = set(AUCELL[m].columns)
    FULL_TF_LISTS[m] = sorted(tf for tf, n in sizes.items() if n >= MIN_TARGETS and tf in aucell_tfs)
    print(f'{m}: {len(FULL_TF_LISTS[m])} TFs (target>={MIN_TARGETS})')

INTERSECT_TFS = sorted(set(FULL_TF_LISTS['GRNBoost2']) & set(FULL_TF_LISTS['SpaGRN']) & set(FULL_TF_LISTS['SpaGTL']))
print(f'\n3-way intersect TF: {len(INTERSECT_TFS)}')
print(INTERSECT_TFS)


---

# Part 1. Allen ISH Comparison — Broad Region (CTX/CNU/IB)

Region-level comparison against Allen ISH `expression_energy`, using Allen's official
CTX/CNU/IB structure IDs (688/623/1129, graph_id=1) instead of unioning sub-structures manually.
With only 3 regions, individual-TF Spearman tests are structurally underpowered (max 3!=6 permutations,
theoretical minimum p≈0.17) — results here are **supportive evidence, not definitive validation**.


## 1.0 Supplementary: Sequencing-depth diagnostic (no correction applied)

In [ ]:
depth_rows = []
for m in METHOD_ORDER:
    for tf in AUCELL[m].columns:
        r, p = pearsonr(AUCELL[m][tf].to_numpy(dtype=float), log_total_counts)
        depth_rows.append(dict(method=m, TF=tf, depth_pearson_r=r, depth_p=p))
depth_df = pd.DataFrame(depth_rows)
depth_df.to_csv(OUT_P1_DEPTH / 'TABLE_bin_depth_diagnostic.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.boxplot(data=depth_df, x='method', y='depth_pearson_r', order=METHOD_ORDER,
            palette=METHOD_COLORS, showfliers=False, ax=ax, linewidth=1.2)
for patch in ax.patches:
    patch.set_edgecolor('black')
ax.axhline(0.3, color='crimson', linestyle='--', linewidth=1, label='|r|=0.3 threshold')
ax.axhline(-0.3, color='crimson', linestyle='--', linewidth=1)
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('')
ax.set_ylabel('AUCell vs log1p(total_counts) (Pearson r)', fontsize=12, fontweight='bold')
ax.set_title('Supplementary: Sequencing-depth diagnostic (no correction applied)', fontsize=13, fontweight='bold')
ax.tick_params(axis='both', labelsize=11)
for lbl in ax.get_xticklabels():
    lbl.set_fontweight('bold')
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(OUT_P1_DEPTH / 'FIG_bin_depth_diagnostic_boxplot.png', dpi=200, bbox_inches='tight')
plt.show()

n_flagged = depth_df.groupby('method').apply(lambda d: (d['depth_pearson_r'].abs() > 0.3).mean() * 100)
print('Depth-confounded regulon 비율 (|r|>0.3), diagnostic only -- no correction applied:')
print(n_flagged.round(1))


## 1.1 Allen broad-region matrix (CTX/CNU/IB structure IDs, cached)

In [ ]:
ALL_UNIQUE_TFS = sorted(set(FULL_TF_LISTS['GRNBoost2']) | set(FULL_TF_LISTS['SpaGRN']) | set(FULL_TF_LISTS['SpaGTL']))
print(f'Total unique TFs across all 3 methods\' full lists: {len(ALL_UNIQUE_TFS)}')

_allen_cache = BASE / 'spatial_method_comparison_no_intersect' / 'FINAL_allen_matrix_BROAD_CTX_CNU_IB.csv'
if _allen_cache.exists():
    print(f'Reusing cached Allen broad-region matrix: {_allen_cache}')
    allen_broad = pd.read_csv(_allen_cache, index_col=0)
else:
    raise FileNotFoundError(
        'Allen broad-region matrix not cached. Re-run build_allen_matrix() with BROAD_STRUCTURE_IDS '
        '(CTX=688, CNU=623, IB=1129, graph_id=1) against the Allen API to regenerate.'
    )

print(f'Allen broad matrix: {allen_broad.shape}')


## 1.2 Region-level Spearman + Pearson tie-break, per method's full regulon set

In [ ]:
def build_pseudobulk_broad(method, tf_list):
    valid_tfs = [tf for tf in tf_list if tf in AUCELL[method].columns]
    sub = AUCELL[method][valid_tfs].copy()
    sub['region'] = broad_region.values
    sub = sub[sub['region'].isin(REGIONS)]
    return sub.groupby('region', observed=True).mean().reindex(REGIONS)


def minmax(x):
    x = np.asarray(x, dtype=float)
    rng = x.max() - x.min()
    return (x - x.min()) / rng if rng > 0 else np.zeros_like(x)


def permutation_test(x, y, n_perm=N_PERM, seed=0):
    rng = np.random.default_rng(seed)
    x, y = np.asarray(x), np.asarray(y)
    obs = spearmanr(x, y)[0]
    if np.isnan(obs):
        return obs, np.nan
    count = sum(1 for _ in range(n_perm) if abs(spearmanr(x, rng.permutation(y))[0]) >= abs(obs))
    return obs, (count + 1) / (n_perm + 1)


MY_MATRICES = {m: build_pseudobulk_broad(m, FULL_TF_LISTS[m]) for m in METHOD_ORDER}

results = []
for m in METHOD_ORDER:
    my_df = MY_MATRICES[m]
    common_tfs = [tf for tf in FULL_TF_LISTS[m] if tf in allen_broad.index]
    for tf in common_tfs:
        x = my_df[tf].values
        y = allen_broad.loc[tf, REGIONS].values.astype(float)
        if np.isnan(x).any() or np.isnan(y).any():
            continue
        rho, p_asymp = spearmanr(x, y)
        _, p_perm = permutation_test(x, y)
        r_pearson, _ = pearsonr(minmax(x), minmax(y))
        results.append(dict(method=m, TF=tf, spearman_rho=rho, p_asymptotic=p_asymp,
                             p_permutation=p_perm, pearson_r=r_pearson))

results_df = pd.DataFrame(results)
results_df.to_csv(OUT_P1 / 'TABLE_spearman_pearson_broad_full_TF.csv', index=False)
print(f'Total (method, TF) tests: {len(results_df)}')
print(results_df.groupby('method')[['spearman_rho', 'pearson_r']].median())


### 1.2.1 Perfect Spearman match rate per method (rho = 1.0)

In [ ]:
stats = []
for m in METHOD_ORDER:
    sub = results_df[results_df.method == m]
    n = len(sub)
    n_perfect = (sub['spearman_rho'] == 1.0).sum()
    stats.append(dict(method=m, n=n, n_perfect=n_perfect, pct=n_perfect/n*100))
stat_df = pd.DataFrame(stats)

fig, ax = plt.subplots(figsize=(7.5, 6))
bars = ax.bar(stat_df['method'], stat_df['pct'], color=[METHOD_COLORS[m] for m in stat_df['method']],
              edgecolor='black', linewidth=1.2)
for i, row in stat_df.iterrows():
    ax.text(i, row['pct'] + 1.5, f"{row['n_perfect']}/{row['n']}\n({row['pct']:.1f}%)",
            ha='center', fontsize=13, fontweight='bold')
ax.set_ylabel('% of TFs with perfect region-rank match (rho = 1.0)', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(stat_df['pct']) * 1.3)
ax.tick_params(axis='both', labelsize=12)
for label in ax.get_xticklabels():
    label.set_fontweight('bold')
ax.set_title('Perfect Spearman match vs Allen ISH\n(broad region: CTX/CNU/IB, full regulon set)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_P1 / 'FIG_perfect_match_by_method.png', dpi=200, bbox_inches='tight')
plt.show()
print(stat_df)


### 1.2.2 TF-level meta-test: is each method's rho distribution shifted above 0? (Wilcoxon signed-rank)

In [ ]:
wilcoxon_rows = []
for m in METHOD_ORDER:
    sub = results_df[results_df.method == m]['spearman_rho'].dropna()
    stat, p = wilcoxon(sub, alternative='greater')
    wilcoxon_rows.append(dict(method=m, n_TFs=len(sub), n_perfect_match=(sub == 1.0).sum(),
                               pct_perfect_match=f'{(sub == 1.0).mean()*100:.1f}%', wilcoxon_p=f'{p:.2e}'))
wilcoxon_table = pd.DataFrame(wilcoxon_rows)
wilcoxon_table.to_csv(OUT_P1 / 'TABLE_perfect_match_and_pvalue.csv', index=False)
print(wilcoxon_table.to_string(index=False))


## 1.3 Intersect TFs (n=22) — per-TF ranking with Spearman + Pearson tie-break

In [ ]:
def build_pseudobulk_broad_tf(method, tf):
    sub = pd.DataFrame({tf: AUCELL[method][tf], 'region': broad_region.values})
    return sub.groupby('region', observed=True)[tf].mean().reindex(REGIONS)


intersect_rows = []
for tf in INTERSECT_TFS:
    if tf not in allen_broad.index:
        continue
    allen_vals = allen_broad.loc[tf, REGIONS].values.astype(float)
    if np.isnan(allen_vals).any():
        continue
    row = dict(TF=tf, Allen_ranking='>'.join(allen_broad.loc[tf, REGIONS].sort_values(ascending=False).index.tolist()))
    for m in METHOD_ORDER:
        my_vals = build_pseudobulk_broad_tf(m, tf).values.astype(float)
        rho, _ = spearmanr(my_vals, allen_vals)
        r_pearson, _ = pearsonr(minmax(my_vals), minmax(allen_vals))
        my_order = '>'.join(build_pseudobulk_broad_tf(m, tf).sort_values(ascending=False).index.tolist())
        row[f'{m}_ranking'] = my_order
        row[f'{m}_rho'] = rho
        row[f'{m}_pearson_r'] = r_pearson
    intersect_rows.append(row)

intersect_table = pd.DataFrame(intersect_rows)


def rank_row_competition(row):
    # Competition ranking (1,1,3): primary key = Spearman rho, tie-break = Pearson r
    keys = [(row[f'{m}_rho'], row[f'{m}_pearson_r'], m) for m in METHOD_ORDER]
    keys_sorted = sorted(keys, key=lambda x: (-x[0], -x[1]))
    ranks, prev_key, rank = {}, None, 0
    for i, (rho, r, m) in enumerate(keys_sorted):
        key = (rho, r)
        if key != prev_key:
            rank = i + 1
        ranks[m] = rank
        prev_key = key
    return ranks


rank_rows = []
for _, row in intersect_table.iterrows():
    ranks = rank_row_competition(row)
    min_rank = min(ranks.values())
    best_methods = ','.join(sorted(m for m, r in ranks.items() if r == min_rank))
    rank_rows.append(dict(TF=row['TF'], best_methods=best_methods,
                           rank_GRNBoost2=ranks['GRNBoost2'], rank_SpaGRN=ranks['SpaGRN'], rank_SpaGTL=ranks['SpaGTL']))
rank_table = pd.DataFrame(rank_rows)

intersect_table = intersect_table.merge(rank_table, on='TF')
intersect_table.to_csv(OUT_P1 / 'TABLE_intersectTF_ranking_tiebreak.csv', index=False)
print(f'{len(intersect_table)} intersect TFs with Allen data')
intersect_table[['TF', 'GRNBoost2_rho', 'SpaGRN_rho', 'SpaGTL_rho', 'best_methods']]


### 1.3.1 Per-TF method ranking bar plot (competition ranking, ties allowed)

In [ ]:
rank_counts = pd.DataFrame({
    m: intersect_table[f'rank_{m}'].value_counts().reindex([1, 2, 3], fill_value=0).astype(int)
    for m in METHOD_ORDER
}).T
rank_counts.columns = ['1st (or tied-1st)', '2nd', '3rd (or tied-3rd)']
rank_counts.to_csv(OUT_P1 / 'TABLE_ranking_counts_tiebreak.csv')

n_total = len(intersect_table)
fig, ax = plt.subplots(figsize=(9, 6.5))
bottom = [0]*len(METHOD_ORDER)
colors = {'1st (or tied-1st)': '#2ca02c', '2nd': '#ffbf00', '3rd (or tied-3rd)': '#d62728'}
for col in rank_counts.columns:
    vals = rank_counts[col].values
    bars = ax.bar(METHOD_ORDER, vals, bottom=bottom, label=col, color=colors[col], edgecolor='black', linewidth=0.8)
    for j, v in enumerate(vals):
        if v > 0:
            ax.text(j, bottom[j] + v/2, str(v), ha='center', va='center', fontsize=14, fontweight='bold')
    bottom = [b+v for b, v in zip(bottom, vals)]

ax.set_ylabel(f'Number of shared TFs with Allen data (n = {n_total})', fontsize=13, fontweight='bold')
ax.set_title('Per-TF method ranking against Allen ISH\n(broad region, intersect TFs, Spearman + Pearson tie-break)', fontsize=14, fontweight='bold')
ax.tick_params(axis='both', labelsize=12)
for label in ax.get_xticklabels():
    label.set_fontweight('bold')
ax.legend(fontsize=11)
plt.tight_layout()
fig.savefig(OUT_P1 / 'FIG_ranking_barplot_intersectTF.png', dpi=200, bbox_inches='tight')
plt.show()
print(rank_counts)


## 1.4 Intersect TF spatial figures

For each intersect TF (with Allen data): 3x3 grid (rows=method, cols=region CTX/CNU/IB; region spots
colored by AUCell percentile rank, rest gray) + region-level bar histograms (GRNBoost2/SpaGRN/SpaGTL/Allen)
below. Method labels show Spearman rho + Pearson r (tie-break); `★ best` marks the competition-ranking
winner(s) — multiple methods can be tied for best.


In [ ]:
EXPR_CMAP = LinearSegmentedColormap.from_list(
    "expr_original", ["#2f9e83", "#dae7e0", "#f0f1e9", "#f2e5d6", "#e8703a", "#c0392b"])


def to_pct_rank(y):
    order = np.argsort(np.argsort(y))
    return order / max(len(y) - 1, 1)


def best_method_set(tf):
    row = intersect_table[intersect_table.TF == tf].iloc[0]
    rhos = {m: row[f'{m}_rho'] for m in METHOD_ORDER}
    pearsons = {m: row[f'{m}_pearson_r'] for m in METHOD_ORDER}
    keys = {m: (rhos[m], pearsons[m]) for m in METHOD_ORDER}
    best_key = max(keys.values())
    best_set = {m for m, k in keys.items() if k == best_key}
    return best_set, rhos, pearsons


def plot_intersect_tf_figure(tf):
    fig = plt.figure(figsize=(15, 17))
    gs_top = fig.add_gridspec(3, 3, left=0.07, right=0.98, top=0.95, bottom=0.55, hspace=0.08, wspace=0.08)
    gs_bottom = fig.add_gridspec(1, 3, left=0.07, right=0.98, top=0.47, bottom=0.06, wspace=0.15)

    best_set, rhos, pearsons = best_method_set(tf)

    for i, m in enumerate(METHOD_ORDER):
        auc_full = AUCELL[m][tf].to_numpy() if tf in AUCELL[m].columns else None
        star = '  \u2605 best' if m in best_set else ''
        for j, r in enumerate(REGIONS):
            ax = fig.add_subplot(gs_top[i, j])
            mask_r = (broad_region == r).values
            ax.scatter(coords[~mask_r, 0], coords[~mask_r, 1], color='#e5e5e5', s=2, linewidths=0)
            if auc_full is not None:
                vals = to_pct_rank(auc_full[mask_r])
                ax.scatter(coords[mask_r, 0], coords[mask_r, 1], c=vals, cmap=EXPR_CMAP, s=3, linewidths=0, vmin=0, vmax=1, alpha=0.9)
            ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal'); ax.invert_yaxis()
            if j == 0:
                ax.set_ylabel(f'{m}{star}', fontsize=12, fontweight='bold', color=METHOD_COLORS[m],
                               rotation=0, ha='right', va='center', labelpad=10)
            if i == 0:
                ax.set_title(r, fontsize=14, fontweight='bold')
            for spine in ax.spines.values():
                spine.set_edgecolor(METHOD_COLORS[m]); spine.set_linewidth(1.5)

    my_means = {m: {r: np.nanmean(AUCELL[m][tf].to_numpy()[(broad_region == r).values]) if tf in AUCELL[m].columns else np.nan
                     for r in REGIONS} for m in METHOD_ORDER}
    global_my_max = max(v for m in METHOD_ORDER for v in my_means[m].values() if not np.isnan(v))
    allen_vals = {r: allen_broad.loc[tf, r] if tf in allen_broad.index else np.nan for r in REGIONS}
    allen_max = max(v for v in allen_vals.values() if not np.isnan(v))

    sources = METHOD_ORDER + ['Allen']
    for col, r in enumerate(REGIONS):
        gs_r = gs_bottom[0, col].subgridspec(len(sources), 1, hspace=0.15)
        for i, src in enumerate(sources):
            ax = fig.add_subplot(gs_r[i, 0])
            if src == 'Allen':
                ax.bar([0], [allen_vals[r]], color=METHOD_COLORS['Allen'], width=0.6)
                ax.set_ylim(0, allen_max * 1.15)
            else:
                ax.bar([0], [my_means[src][r]], color=METHOD_COLORS[src], width=0.6)
                ax.set_ylim(0, global_my_max * 1.15)
            ax.set_xticks([])
            if col == 0:
                if src == 'Allen':
                    label = 'Allen'
                else:
                    star_h = '  \u2605 best' if src in best_set else ''
                    label = f'{src}{star_h}\nrho={rhos[src]:.2f}  r={pearsons[src]:.2f}'
                ax.set_ylabel(label, fontsize=9, fontweight='bold', color=METHOD_COLORS[src], rotation=0, ha='right', va='center')
                ax.tick_params(labelsize=8)
            else:
                ax.set_yticks([])
            if i == 0:
                ax.set_title(r, fontsize=13, fontweight='bold')
            for spine in ['top', 'right']:
                ax.spines[spine].set_visible(False)

    fig.suptitle(f'{tf}', fontsize=18, fontweight='bold', y=0.99)
    fname = OUT_P1_TF / f'{tf}.png'
    plt.savefig(fname, dpi=140, bbox_inches='tight')
    plt.show()
    return fname


saved = []
for tf in intersect_table['TF']:
    f = plot_intersect_tf_figure(tf)
    saved.append(f)
print(f'Saved {len(saved)} intersect TF figures to {OUT_P1_TF}')


---

# Part 2. Internal Spatial Validation (Allen-independent)

Moran's I / Geary's C computed on the whole tissue (bin-level, K=5/10/15 nearest-neighbor graph,
999-permutation z-scores). Unlike Part 1, this does **not** depend on `region_proxy_from_bin_class`
or Allen ISH data at all -- it only uses spot coordinates and each method's AUCell activity, so it
covers each method's *entire* regulon set (not just Allen-covered or intersect TFs).


## 2.1 Region label spatial plots (context, no computation)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
for r in sorted(region_proxy.unique()):
    mask = (region_proxy == r).values
    ax.scatter(coords[mask, 0], coords[mask, 1], s=3, color=plt.get_cmap('tab20')(hash(r) % 20 / 20),
               label=f'{r} (n={mask.sum()})', linewidths=0)
ax.set_title('Original region_proxy_from_bin_class', fontsize=15, fontweight='bold')
ax.invert_yaxis(); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
ax.legend(fontsize=9, markerscale=3, loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.savefig(OUT_P2_ORIG / 'spatial_original_region.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
BROAD_COLORS = {
    'CTX': '#d62728', 'CNU': '#1f77b4', 'IB': '#2ca02c',
    'NA_region': '#dddddd', 'Unresolved': '#eeeeee',
    'MB': '#9467bd', 'CB': '#8c564b', 'MY': '#e377c2', 'Pons': '#bcbd22',
}
broad_region_filled = broad_region.fillna(region_proxy)

fig, ax = plt.subplots(figsize=(10, 9))
plot_order = ['CTX', 'CNU', 'IB', 'MB', 'CB', 'MY', 'Pons', 'NA_region', 'Unresolved']
for r in plot_order:
    mask = (broad_region_filled == r).values
    if mask.sum() == 0:
        continue
    ax.scatter(coords[mask, 0], coords[mask, 1], s=3, color=BROAD_COLORS.get(r, 'black'),
               label=f'{r} (n={mask.sum()})', linewidths=0)
ax.set_title('Broad region (Allen official hierarchy: CTX/CNU/IB)', fontsize=15, fontweight='bold')
ax.invert_yaxis(); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
ax.legend(fontsize=11, markerscale=3, loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.savefig(OUT_P2_BROAD / 'spatial_broad_region_CTX_CNU_IB.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.2 KNN graph + Moran's I / Geary's C utilities (K-sweep)

In [ ]:
def build_knn_graph(k):
    nn = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    _, idx = nn.kneighbors(coords)
    idx = idx[:, 1:]
    row = np.repeat(np.arange(n_spots), k)
    col = idx.flatten()
    W = sparse.csr_matrix((np.ones(len(row)), (row, col)), shape=(n_spots, n_spots))
    W = W.maximum(W.T)
    return W, W.sum()


def morans_i_batch(Y, W, S0):
    Z = Y - Y.mean(axis=0, keepdims=True)
    num = np.einsum('ij,ij->j', Z, (W @ Z))
    den = np.einsum('ij,ij->j', Z, Z)
    den[den == 0] = np.nan
    return (n_spots / S0) * (num / den)


def gearys_c_batch(Y, W, S0):
    Z = Y - Y.mean(axis=0, keepdims=True)
    den = np.einsum('ij,ij->j', Z, Z)
    den[den == 0] = np.nan
    deg = np.asarray(W.sum(axis=1)).flatten()
    quad = np.einsum('ij,ij->j', Y, (W @ Y))
    sq_sum = (deg[:, None] * (Y ** 2)).sum(axis=0)
    num = 2 * (sq_sum - quad)
    return ((n_spots - 1) / (2 * S0)) * (num / den)


def permutation_zscores(activity, W, S0, n_perm=N_PERM, seed=RANDOM_SEED):
    obs_I = morans_i_batch(activity, W, S0)
    obs_C = gearys_c_batch(activity, W, S0)
    rng = np.random.default_rng(seed)
    null_I = np.empty((n_perm, activity.shape[1]))
    null_C = np.empty((n_perm, activity.shape[1]))
    for p in range(n_perm):
        perm_idx = rng.permutation(n_spots)
        Y = activity[perm_idx]
        null_I[p] = morans_i_batch(Y, W, S0)
        null_C[p] = gearys_c_batch(Y, W, S0)
    z_moran = (obs_I - null_I.mean(axis=0)) / null_I.std(axis=0)
    z_geary = -(obs_C - null_C.mean(axis=0)) / null_C.std(axis=0)
    return z_moran, z_geary


print('KNN + Moran/Geary utilities ready.')


## 2.3 K-sweep computation — all TFs per method (Allen-independent, whole tissue)

In [ ]:
_ksweep_cache = BASE / 'spatial_method_comparison_no_intersect' / 'PART2_moran_geary_KSWEEP_all_TF.csv'
if _ksweep_cache.exists():
    print(f'Reusing cached K-sweep Moran/Geary results: {_ksweep_cache}')
    moran_ksweep_df = pd.read_csv(_ksweep_cache)
else:
    rows = []
    for k in K_VALUES:
        print(f'=== K={k} ===')
        W, S0 = build_knn_graph(k)
        for m in METHOD_ORDER:
            tfs = FULL_TF_LISTS[m]
            activity = AUCELL[m][tfs].to_numpy(dtype=np.float32)
            print(f'{m}: computing for {len(tfs)} TFs (999 perms)...')
            z_moran, z_geary = permutation_zscores(activity, W, S0)
            for i, tf in enumerate(tfs):
                rows.append(dict(K=k, method=m, TF=tf, z_moran=z_moran[i], z_geary=z_geary[i]))
        print(f'K={k} done')
    moran_ksweep_df = pd.DataFrame(rows)
    moran_ksweep_df.to_csv(_ksweep_cache, index=False)

moran_ksweep_df['set'] = moran_ksweep_df['TF'].apply(lambda t: 'Intersect' if t in INTERSECT_TFS else 'Non-intersect')
print(f'{len(moran_ksweep_df)} rows ({moran_ksweep_df.TF.nunique()} unique TFs x {len(K_VALUES)} K values, summed across methods)')


### 2.3.1 K-sweep boxplots — Moran's I and Geary's C, separate figures, K panels side-by-side

In [ ]:
def make_ksweep_boxplot(sub_df, metric, ylabel, title, out_path):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    for ax, k in zip(axes, K_VALUES):
        sub_k = sub_df[sub_df.K == k]
        sns.boxplot(data=sub_k, x='method', y=metric, order=METHOD_ORDER,
                    palette=METHOD_COLORS, showfliers=False, ax=ax, linewidth=1.2)
        for patch in ax.patches:
            patch.set_edgecolor('black')
        ax.axhline(0, color='gray', linestyle=':', linewidth=1)
        ax.set_xlabel('')
        ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
        ns = sub_k.groupby('method')['TF'].nunique().reindex(METHOD_ORDER)
        ax.set_title(f'K={k}  (n: {", ".join(f"{m}={ns[m]}" for m in METHOD_ORDER)})', fontsize=12, fontweight='bold')
        ax.tick_params(axis='both', labelsize=11)
        for lbl in ax.get_xticklabels():
            lbl.set_fontweight('bold')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()


sub_intersect = moran_ksweep_df[moran_ksweep_df['set'] == 'Intersect']
sub_non = moran_ksweep_df[moran_ksweep_df['set'] == 'Non-intersect']

make_ksweep_boxplot(sub_intersect, 'z_moran', "Moran's I z-score",
                     "Moran's I -- Intersect TFs (n=22), K-sweep",
                     OUT_P1_BOXPLOTS / 'BOXPLOT_moran_intersectTF_Ksweep.png')
make_ksweep_boxplot(sub_intersect, 'z_geary', "Geary's C z-score (sign-flipped)",
                     "Geary's C -- Intersect TFs (n=22), K-sweep",
                     OUT_P1_BOXPLOTS / 'BOXPLOT_geary_intersectTF_Ksweep.png')
make_ksweep_boxplot(sub_non, 'z_moran', "Moran's I z-score",
                     "Moran's I -- Non-intersect TFs, K-sweep",
                     OUT_P1_BOXPLOTS / 'BOXPLOT_moran_nonintersectTF_Ksweep.png')
make_ksweep_boxplot(sub_non, 'z_geary', "Geary's C z-score (sign-flipped)",
                     "Geary's C -- Non-intersect TFs, K-sweep",
                     OUT_P1_BOXPLOTS / 'BOXPLOT_geary_nonintersectTF_Ksweep.png')


### 2.3.2 Summary: % of TFs significantly spatially organized per method (K=10, |z|>1.96)

In [ ]:
summary_k10 = moran_ksweep_df[moran_ksweep_df.K == 10].groupby('method').apply(lambda d: pd.Series({
    'n_TF': len(d),
    'n_sig_moran': (d['z_moran'].abs() > 1.96).sum(),
    'pct_sig_moran': (d['z_moran'].abs() > 1.96).mean() * 100,
    'median_z_moran': d['z_moran'].median(),
}), include_groups=False)
summary_k10.to_csv(OUT_P2 / 'TABLE_moran_summary_K10_by_method.csv')
print(summary_k10)


## 2.4 Intersect TF (n=22) spatial figures with z_moran / z_geary (K=10, whole tissue)

In [ ]:
def plot_intersect_tf_moran_figure(tf):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
    for ax, m in zip(axes, METHOD_ORDER):
        auc_full = AUCELL[m][tf].to_numpy() if tf in AUCELL[m].columns else None
        if auc_full is not None:
            vals = to_pct_rank(auc_full)
            ax.scatter(coords[:, 0], coords[:, 1], c=vals, cmap=EXPR_CMAP, s=3, linewidths=0, vmin=0, vmax=1, alpha=0.9)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal'); ax.invert_yaxis()

        row = moran_ksweep_df[(moran_ksweep_df.method == m) & (moran_ksweep_df.TF == tf) & (moran_ksweep_df.K == 10)]
        if len(row):
            z_moran = row['z_moran'].values[0]
            z_geary = row['z_geary'].values[0]
            title = f'{m}\nz_moran={z_moran:.1f}  z_geary={z_geary:.1f}'
        else:
            title = f'{m}\n(no data)'
        ax.set_title(title, fontsize=13, fontweight='bold', color=METHOD_COLORS[m])
        for spine in ax.spines.values():
            spine.set_edgecolor(METHOD_COLORS[m]); spine.set_linewidth(1.5)

    fig.suptitle(f'{tf}  (whole tissue, K=10)', fontsize=17, fontweight='bold')
    plt.tight_layout()
    fname = OUT_P2_TF / f'{tf}.png'
    plt.savefig(fname, dpi=140, bbox_inches='tight')
    plt.show()
    return fname


saved_moran = []
for tf in INTERSECT_TFS:
    f = plot_intersect_tf_moran_figure(tf)
    saved_moran.append(f)
print(f'Saved {len(saved_moran)} intersect TF Moran figures to {OUT_P2_TF}')


---

## Overall Summary

- **Part 1 (Allen, broad region CTX/CNU/IB)**: Region-level Spearman + Pearson tie-break rankings are
  supportive evidence only (n=3 regions => structurally underpowered). Perfect-match rate and the
  TF-level Wilcoxon meta-test (treating each method's full TF set as the sample) both show
  SpaGRN/SpaGTL outperforming GRNBoost2. Sequencing-depth diagnostic is reported as Supplementary
  context only -- no correction is applied to the reported results.
- **Part 2 (internal, Allen-independent)**: Moran's I / Geary's C on the whole tissue (bin-level,
  K=5/10/15) show that nearly all TFs across all three methods are significantly spatially organized,
  but the *strength* of that organization (z-score magnitude) is consistently ordered
  SpaGRN > SpaGTL > GRNBoost2 -- for both the intersect TF set and each method's full regulon set,
  independent of Allen and of region annotation quality.
